In [1]:
# this notebook will scale the data and
# train test split
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

model_name = "BTCUSD-5m"
src_path = folder_path / "clean" / f"{model_name}-v5-D-input.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,...,isLTR,c_EMA50_pct,c_EMA200_pct,close_pct_w_range,c_vwap_96,barsSinceITR,hiDTK_ITR,lowDTK_ITR,closeDTK_ITR,hl_pct
0,1704238800000,44831.761719,104.694832,109.456940,1704238800000,1,1,-0.000508,0.602508,-0.000508,...,0,-0.003719,-0.004638,0.725950,-0.004909,32,0.006678,0.004529,0.005196,0.002137
1,1704239100000,44830.718750,77.087044,107.552177,1704239100000,1,1,-0.000023,0.446939,-0.000023,...,0,-0.003595,-0.004615,0.725677,-0.004858,33,0.005412,0.004643,0.005173,0.000765
2,1704239400000,44957.578125,152.559433,107.905785,1704239400000,0,0,0.002830,0.884560,0.002830,...,0,-0.000734,-0.001763,0.758855,-0.001950,34,0.008581,0.004883,0.008017,0.003669
3,1704239700000,44946.910156,43.273781,106.447174,1704239700000,1,1,-0.000237,0.253292,-0.000237,...,0,-0.000933,-0.001980,0.756065,-0.002137,35,0.008363,0.007268,0.007778,0.001086
4,1704240000000,44955.511719,137.393387,107.078873,1704240000000,0,0,0.000191,0.806874,0.000191,...,0,-0.000713,-0.001771,0.758314,-0.001887,36,0.009802,0.007778,0.007971,0.002008


In [2]:
# df = df.drop(columns=["timestamp"])
print(df.shape)
print(df.columns)
df.head()

(228101, 24)
Index(['timestamp', 'close', 'vol', 'atr42', 'ts_5m', 'label', 'train_mask',
       'body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', 'ts_15m', 'isSTR',
       'isITR', 'isLTR', 'c_EMA50_pct', 'c_EMA200_pct', 'close_pct_w_range',
       'c_vwap_96', 'barsSinceITR', 'hiDTK_ITR', 'lowDTK_ITR', 'closeDTK_ITR',
       'hl_pct'],
      dtype='object')


,timestamp,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,...,isLTR,c_EMA50_pct,c_EMA200_pct,close_pct_w_range,c_vwap_96,barsSinceITR,hiDTK_ITR,lowDTK_ITR,closeDTK_ITR,hl_pct
0,1704238800000,44831.761719,104.694832,109.456940,1704238800000,1,1,-0.000508,0.602508,-0.000508,...,0,-0.003719,-0.004638,0.725950,-0.004909,32,0.006678,0.004529,0.005196,0.002137
1,1704239100000,44830.718750,77.087044,107.552177,1704239100000,1,1,-0.000023,0.446939,-0.000023,...,0,-0.003595,-0.004615,0.725677,-0.004858,33,0.005412,0.004643,0.005173,0.000765
2,1704239400000,44957.578125,152.559433,107.905785,1704239400000,0,0,0.002830,0.884560,0.002830,...,0,-0.000734,-0.001763,0.758855,-0.001950,34,0.008581,0.004883,0.008017,0.003669
3,1704239700000,44946.910156,43.273781,106.447174,1704239700000,1,1,-0.000237,0.253292,-0.000237,...,0,-0.000933,-0.001980,0.756065,-0.002137,35,0.008363,0.007268,0.007778,0.001086
4,1704240000000,44955.511719,137.393387,107.078873,1704240000000,0,0,0.000191,0.806874,0.000191,...,0,-0.000713,-0.001771,0.758314,-0.001887,36,0.009802,0.007778,0.007971,0.002008


In [3]:
# tactical drop extra timeframe and non-stationary data
# prepare for scaler : scaler is order strict
df.drop(columns=["close", "vol", "atr42","ts_5m", "ts_15m"], inplace=True)
print(df.shape)
df.head()

(228101, 19)


,timestamp,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,isSTR,isITR,isLTR,c_EMA50_pct,c_EMA200_pct,close_pct_w_range,c_vwap_96,barsSinceITR,hiDTK_ITR,lowDTK_ITR,closeDTK_ITR,hl_pct
0,1704238800000,1,1,-0.000508,0.602508,-0.000508,0.002442,-1,0,0,-0.003719,-0.004638,0.725950,-0.004909,32,0.006678,0.004529,0.005196,0.002137
1,1704239100000,1,1,-0.000023,0.446939,-0.000023,0.002399,0,0,0,-0.003595,-0.004615,0.725677,-0.004858,33,0.005412,0.004643,0.005173,0.000765
2,1704239400000,0,0,0.002830,0.884560,0.002830,0.002400,0,0,0,-0.000734,-0.001763,0.758855,-0.001950,34,0.008581,0.004883,0.008017,0.003669
3,1704239700000,1,1,-0.000237,0.253292,-0.000237,0.002368,0,0,0,-0.000933,-0.001980,0.756065,-0.002137,35,0.008363,0.007268,0.007778,0.001086
4,1704240000000,0,0,0.000191,0.806874,0.000191,0.002382,0,0,0,-0.000713,-0.001771,0.758314,-0.001887,36,0.009802,0.007778,0.007971,0.002008


# At this point

- hl_pct is an extra

# Prepare input and output
```
Label balance (trainable only):
  Up   ( 1) : 76,205  (49.5%)
  Down (-1) : 77,788  (50.5%)
  Total     : 153,993  (67.3% of clean rows)
  Timeout   : 74,684
```

In [4]:
# Sort chronologically first — never shuffle
df = df.sort_values('timestamp').reset_index(drop=True)
n = len(df)

# Chronological split on ALL rows (includes timeout rows — preserves time boundaries)
train_all = df.iloc[:int(n*0.8)]
val_all   = df.iloc[int(n*0.8):int(n*0.9)]
test_all  = df.iloc[int(n*0.9):]

# Apply train_mask: filter to trainable rows (label 1 or -1) within each split
train_df = train_all[train_all['train_mask'] == 1].copy()
val_df   = val_all[val_all['train_mask'] == 1].copy()
test_df  = test_all[test_all['train_mask'] == 1].copy()

print(f"All rows  — Train: {len(train_all):,} | Val: {len(val_all):,} | Test: {len(test_all):,}")
print(f"Trainable — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


All rows  — Train: 182,480 | Val: 22,810 | Test: 22,811
Trainable — Train: 123,269 | Val: 15,307 | Test: 15,007


# However — there's a catch for your case. You're using a chronological split

In [5]:
print(df.columns)
print(len(df.columns))
del df

Index(['timestamp', 'label', 'train_mask', 'body_pct', 'vol_norm',
       'close_ret1', 'atr42_pct', 'isSTR', 'isITR', 'isLTR', 'c_EMA50_pct',
       'c_EMA200_pct', 'close_pct_w_range', 'c_vwap_96', 'barsSinceITR',
       'hiDTK_ITR', 'lowDTK_ITR', 'closeDTK_ITR', 'hl_pct'],
      dtype='object')
19


In [6]:
# apply scaler
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import date

formatted_date = date.today().strftime("%Y%m")

# Drop redundant timestamp alias
# train_df = train_df.drop(columns=["ts_5m"])
# val_df   = val_df.drop(columns=["ts_5m"])
# test_df  = test_df.drop(columns=["ts_5m"])

# Scale all features except non-numeric / target columns
# ex 'isSTR', 'isITR', 'isLTR'
scale_cols = ['body_pct', 'vol_norm',
       'close_ret1', 'atr42_pct', 'c_EMA50_pct',
       'c_EMA200_pct', 'close_pct_w_range', 'c_vwap_96', 'barsSinceITR',
       'hiDTK_ITR', 'lowDTK_ITR', 'closeDTK_ITR', 'hl_pct']

scaler = StandardScaler()
train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])  # fit + transform on train only
val_df[scale_cols]   = scaler.transform(val_df[scale_cols])        # transform only
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])       # transform only

joblib.dump(scaler, folder_path / "scaler" / f"{formatted_date}-{model_name}-scaler-D.pkl")
print("Scaler saved.")
print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")


Scaler saved.
Train: (123269, 19) | Val: (15307, 19) | Test: (15007, 19)


In [7]:
# save train data
# where Y = 0 around 26 cells, remove them during training
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train-D.jsonl"
val_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-val-D.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test-D.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
val_df.to_json(val_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved val: {val_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-BTCUSD-5m-train-D.jsonl
Saved val: 202603-BTCUSD-5m-val-D.jsonl
Saved test:  202603-BTCUSD-5m-test-D.jsonl


In [8]:
print(train_df.columns)
print(len(train_df.columns))
print(scale_cols)
print(len(scale_cols))

Index(['timestamp', 'label', 'train_mask', 'body_pct', 'vol_norm',
       'close_ret1', 'atr42_pct', 'isSTR', 'isITR', 'isLTR', 'c_EMA50_pct',
       'c_EMA200_pct', 'close_pct_w_range', 'c_vwap_96', 'barsSinceITR',
       'hiDTK_ITR', 'lowDTK_ITR', 'closeDTK_ITR', 'hl_pct'],
      dtype='object')
19
['body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', 'c_EMA50_pct', 'c_EMA200_pct', 'close_pct_w_range', 'c_vwap_96', 'barsSinceITR', 'hiDTK_ITR', 'lowDTK_ITR', 'closeDTK_ITR', 'hl_pct']
13
